# Distribuicao espacial das redes pluviometricas

Este notebook visualiza, de forma independente, as estacoes AlertaRio e WebSirene mapeadas para a grade do Radar Sumare. O mapa ajuda a avaliar cobertura espacial e sobreposicoes potenciais antes de qualquer integracao de observacoes WebSirene ao pipeline de treinamento.

> A presenca no mapa nao certifica a qualidade da serie WebSirene. A selecao de estacoes para treinamento deve ocorrer somente apos a etapa de controle de qualidade.

In [ ]:
from html import escape
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for directory in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (directory / 'data' / 'sumare_radar_latlon_grid.npz').is_file():
            return directory
    raise FileNotFoundError('Execute o notebook a partir do repositorio ou de notebooks/.')


PROJECT_ROOT = find_project_root()
ALERTARIO_PATH = PROJECT_ROOT / 'data' / 'mapeamento_pixel_estacao_alertario.csv'
WEBSIRENE_PATH = PROJECT_ROOT / 'data' / 'mapeamento_pixel_estacao.csv'
GRID_PATH = PROJECT_ROOT / 'data' / 'sumare_radar_latlon_grid.npz'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'analysis' / 'geospatial'
FOLIUM_PATH = OUTPUT_DIR / 'mapa_redes_pluviometricas.html'
FIGURE_PATH = OUTPUT_DIR / 'mapa_redes_pluviometricas.png'

for path in (ALERTARIO_PATH, WEBSIRENE_PATH, GRID_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Arquivo ausente: {path}')

print('Projeto:', PROJECT_ROOT)
print('AlertaRio:', ALERTARIO_PATH)
print('WebSirene:', WEBSIRENE_PATH)

In [ ]:
REQUIRED_COLUMNS = {'station_id', 'nome', 'latitude', 'longitude', 'pixel_i', 'pixel_j'}
TARGET_HEIGHT = 128
TARGET_WIDTH = 128
HEIGHT_ORIG = 656
WIDTH_ORIG = 654
CROP_MARGIN_PIXELS = 20


def decode_name(value: object) -> str:
    text = str(value)
    if 'Ã' not in text:
        return text
    try:
        return text.encode('latin1').decode('utf-8')
    except UnicodeError:
        return text


def load_mapping(path: Path, source: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    missing = REQUIRED_COLUMNS - set(frame.columns)
    if missing:
        raise ValueError(f'{source}: colunas ausentes: {sorted(missing)}')
    frame = frame.loc[:, sorted(REQUIRED_COLUMNS)].copy()
    frame['source'] = source
    frame['nome'] = frame['nome'].map(decode_name)
    frame['pixel_i_128'] = np.rint(frame['pixel_i'] * TARGET_HEIGHT / HEIGHT_ORIG).astype(int).clip(0, TARGET_HEIGHT - 1)
    frame['pixel_j_128'] = np.rint(frame['pixel_j'] * TARGET_WIDTH / WIDTH_ORIG).astype(int).clip(0, TARGET_WIDTH - 1)
    return frame


alertario = load_mapping(ALERTARIO_PATH, 'AlertaRio')
websirene = load_mapping(WEBSIRENE_PATH, 'WebSirene')
stations = pd.concat((alertario, websirene), ignore_index=True)
CROP_TOP = max(0, int(alertario['pixel_i_128'].min()) - CROP_MARGIN_PIXELS)
CROP_BOTTOM = min(TARGET_HEIGHT, int(alertario['pixel_i_128'].max()) + CROP_MARGIN_PIXELS + 1)
CROP_LEFT = max(0, int(alertario['pixel_j_128'].min()) - CROP_MARGIN_PIXELS)
CROP_RIGHT = min(TARGET_WIDTH, int(alertario['pixel_j_128'].max()) + CROP_MARGIN_PIXELS + 1)

with np.load(GRID_PATH) as grid:
    latitude_grid = grid['lat']
    longitude_grid = grid['lon']

display(stations.groupby('source').agg(estacoes=('station_id', 'size'), lat_min=('latitude', 'min'), lat_max=('latitude', 'max'), lon_min=('longitude', 'min'), lon_max=('longitude', 'max')))
display(stations.sort_values(['source', 'station_id']).reset_index(drop=True))

In [ ]:
def grid_boundary(latitude: np.ndarray, longitude: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    rows, columns = latitude.shape
    top = np.arange(columns)
    right = np.arange(1, rows)
    bottom = np.arange(columns - 2, -1, -1)
    left = np.arange(rows - 2, 0, -1)
    row = np.concatenate((np.zeros(columns, dtype=int), right, np.full(columns - 1, rows - 1), left))
    column = np.concatenate((top, np.full(rows - 1, columns - 1), bottom, np.zeros(rows - 2, dtype=int)))
    return latitude[row, column], longitude[row, column]


pairs = (
    stations.groupby(['pixel_i_128', 'pixel_j_128'])['source']
    .agg(lambda values: ', '.join(sorted(set(values))))
    .reset_index(name='redes')
    .query("redes == 'AlertaRio, WebSirene'")
)
print(f'Grade geografica: {latitude_grid.shape[0]} x {latitude_grid.shape[1]}')
print(f'ROI cropada do treino AlertaRio: {CROP_TOP}:{CROP_BOTTOM}, {CROP_LEFT}:{CROP_RIGHT}; shape=({CROP_BOTTOM - CROP_TOP}, {CROP_RIGHT - CROP_LEFT})')
print(f'Pixels 128 x 128 ocupados pelas duas redes: {len(pairs)}')
display(pairs)

## Mapa interativo

Use o controle no canto superior direito para exibir ou ocultar cada rede e os nomes das estacoes. O mapa usa OpenStreetMap e nao requer chave de API.

In [ ]:
SHOW_STATION_NAMES = False
RADAR_LATITUDE = -22.955139
RADAR_LONGITUDE = -43.248278
STYLE = {
    'AlertaRio': {'edge': '#b2182b', 'fill': '#ef3b2c', 'label': '#7f0000'},
    'WebSirene': {'edge': '#6a3d9a', 'fill': '#9e77c7', 'label': '#4a1486'},
}

boundary_latitude, boundary_longitude = grid_boundary(latitude_grid, longitude_grid)
row_indices = np.clip(((np.arange(TARGET_HEIGHT) + 0.5) * latitude_grid.shape[0] / TARGET_HEIGHT).astype(int), 0, latitude_grid.shape[0] - 1)
column_indices = np.clip(((np.arange(TARGET_WIDTH) + 0.5) * longitude_grid.shape[1] / TARGET_WIDTH).astype(int), 0, longitude_grid.shape[1] - 1)
latitude_grid_128 = latitude_grid[row_indices][:, column_indices]
longitude_grid_128 = longitude_grid[row_indices][:, column_indices]
crop_latitude, crop_longitude = grid_boundary(latitude_grid_128[CROP_TOP:CROP_BOTTOM, CROP_LEFT:CROP_RIGHT], longitude_grid_128[CROP_TOP:CROP_BOTTOM, CROP_LEFT:CROP_RIGHT])
map_center = [stations['latitude'].mean(), stations['longitude'].mean()]
station_map = folium.Map(location=map_center, zoom_start=10, tiles='OpenStreetMap', control_scale=True)

folium.GeoJson(
    {'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [list(zip(boundary_longitude, boundary_latitude))]}},
    name='Area da grade do Radar Sumare',
    style_function=lambda _: {'color': '#2171b5', 'weight': 2, 'fillColor': '#9ecae1', 'fillOpacity': 0.12},
).add_to(station_map)
folium.GeoJson(
    {'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [list(zip(crop_longitude, crop_latitude))]}},
    name=f'ROI cropada AlertaRio ({CROP_BOTTOM - CROP_TOP} x {CROP_RIGHT - CROP_LEFT})',
    style_function=lambda _: {'color': '#f16913', 'weight': 2, 'dashArray': '6 4', 'fillColor': '#fdae6b', 'fillOpacity': 0.12},
).add_to(station_map)
folium.Marker([RADAR_LATITUDE, RADAR_LONGITUDE], tooltip='Radar Sumare', icon=folium.Icon(color='black', icon='tower', prefix='fa')).add_to(station_map)

layers = {source: folium.FeatureGroup(name=f'Estacoes {source}', show=True).add_to(station_map) for source in STYLE}
name_layers = {source: folium.FeatureGroup(name=f'Nomes {source}', show=SHOW_STATION_NAMES).add_to(station_map) for source in STYLE}
for station in stations.sort_values(['source', 'station_id']).itertuples():
    style = STYLE[station.source]
    popup = (
        f'<b>{escape(str(station.nome))}</b><br>'
        f'Rede: {station.source}<br>'
        f'ID: {station.station_id}<br>'
        f'Latitude: {station.latitude:.5f}<br>'
        f'Longitude: {station.longitude:.5f}<br>'
        f'Pixel 128 x 128: ({station.pixel_i_128}, {station.pixel_j_128})'
    )
    folium.CircleMarker(
        [station.latitude, station.longitude], radius=5, color=style['edge'], weight=2,
        fill=True, fill_color=style['fill'], fill_opacity=0.9, tooltip=f'{station.source}: {station.nome}',
        popup=folium.Popup(popup, max_width=300),
    ).add_to(layers[station.source])
    folium.Marker(
        [station.latitude, station.longitude],
        icon=folium.DivIcon(html=f'<div style="font-size: 10px; color: {style["label"]}; white-space: nowrap;">{escape(str(station.nome))}</div>'),
    ).add_to(name_layers[station.source])

folium.LayerControl(collapsed=False).add_to(station_map)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
station_map.save(FOLIUM_PATH)
print(f'Mapa interativo salvo em: {FOLIUM_PATH}')
station_map

In [ ]:
ANNOTATE_NAMES = False
fig, (geographic_axis, pixel_axis) = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)
geographic_axis.fill(boundary_longitude, boundary_latitude, color='#9ecae1', alpha=0.3, label='Grade do Radar Sumare')
geographic_axis.plot(boundary_longitude, boundary_latitude, color='#2171b5', linewidth=1)
geographic_axis.fill(crop_longitude, crop_latitude, color='#fdae6b', alpha=0.25, label='ROI cropada AlertaRio')
geographic_axis.plot(crop_longitude, crop_latitude, color='#f16913', linewidth=1.5, linestyle='--')

for source, frame in stations.groupby('source'):
    style = STYLE[source]
    geographic_axis.scatter(frame['longitude'], frame['latitude'], color=style['fill'], edgecolors=style['edge'], s=42, zorder=3, label=source)
    pixel_axis.scatter(frame['pixel_j_128'], frame['pixel_i_128'], color=style['fill'], edgecolors=style['edge'], s=42, label=source)
    if ANNOTATE_NAMES:
        for station in frame.itertuples():
            geographic_axis.annotate(station.nome, (station.longitude, station.latitude), xytext=(4, 4), textcoords='offset points', fontsize=7)
            pixel_axis.annotate(station.nome, (station.pixel_j_128, station.pixel_i_128), xytext=(4, 4), textcoords='offset points', fontsize=7)

geographic_axis.scatter([RADAR_LONGITUDE], [RADAR_LATITUDE], color='black', marker='*', s=150, zorder=4, label='Radar Sumare')
geographic_axis.set(title='Redes pluviometricas na area do radar', xlabel='Longitude', ylabel='Latitude')
geographic_axis.set_aspect(1 / np.cos(np.deg2rad(RADAR_LATITUDE)))
geographic_axis.grid(alpha=0.25)
geographic_axis.legend(loc='upper right')

pixel_axis.set(title='Posicoes supervisionadas na grade 128 x 128', xlabel='Coluna', ylabel='Linha', xlim=(-1, TARGET_WIDTH), ylim=(TARGET_HEIGHT, -1))
pixel_axis.add_patch(plt.Rectangle((CROP_LEFT, CROP_TOP), CROP_RIGHT - CROP_LEFT, CROP_BOTTOM - CROP_TOP, fill=False, edgecolor='#f16913', linewidth=1.5, linestyle='--', label='ROI cropada AlertaRio'))
pixel_axis.set_aspect('equal')
pixel_axis.set_xticks(np.arange(0, TARGET_WIDTH + 1, 16))
pixel_axis.set_yticks(np.arange(0, TARGET_HEIGHT + 1, 16))
pixel_axis.grid(alpha=0.35)
pixel_axis.legend(loc='upper right')

fig.savefig(FIGURE_PATH, dpi=200, bbox_inches='tight')
print(f'Figura salva em: {FIGURE_PATH}')
plt.show()

## Proximo uso

A sobreposicao espacial indica apenas candidatos para comparacao entre redes. A etapa seguinte deve medir cobertura temporal, diferencas de precipitacao e regras de qualidade por estacao WebSirene antes de gerar targets ou executar treinamento com essa fonte.